# Limpiar los siniestros
Objetivo: dejar una sola tabla con los siniestros con muertos o heridos de 2015 a 2025, con coordenadas.

In [38]:
import pandas as pd

carpeta = "../data/raw/Siniestros_viales/"

siniestros = pd.read_csv(carpeta + "SIGAT_ANUARIO_2024_Siniestros.csv.gz", low_memory=False)
sin_2025 = pd.read_excel(carpeta + "SIGAT_ANUARIO_2025.xlsx", sheet_name="Siniestros")

In [39]:
columnas = ["Codigo_Accidente", "Fecha_Acc", "AA_Acc", "Hora_Acc", "Localidad",
            "Direccion", "Latitud", "Longitud", "Clase_Acc", "Gravedad_indicador_30d"]

siniestros = siniestros[columnas]
print(siniestros.shape)

(278614, 10)


Arreglar 2025

In [40]:
sin_2025 = sin_2025.rename(columns={"Codigo_Siniestro": "Codigo_Accidente",
                                    "Fecha_Siniestro": "Fecha_Acc",
                                    "AA_Siniestro": "AA_Acc",
                                    "Hora_Siniestro": "Hora_Acc",
                                    "Clase_Siniestro": "Clase_Acc"})
sin_2025 = sin_2025[columnas]

# En 2025 las coordenadas vienen como texto y con coma ("4,58") en vez de punto ("4.58")
sin_2025["Latitud"] = sin_2025["Latitud"].str.replace(",", ".").astype(float)
sin_2025["Longitud"] = sin_2025["Longitud"].str.replace(",", ".").astype(float)
print(sin_2025.shape)

(11573, 10)


Juntar los 2 y revisar repetidos

In [41]:
todos = pd.concat([siniestros, sin_2025], ignore_index=True)
print(todos.shape)
print(todos["Codigo_Accidente"].duplicated().sum())

todos = todos.drop_duplicates(subset="Codigo_Accidente", keep="last")
print(todos.shape)

(290187, 10)
3
(290184, 10)


Quedarnos con los siniestros con víctimas

In [42]:
print(todos["Gravedad_indicador_30d"].value_counts())

con_victimas = todos[todos["Gravedad_indicador_30d"].isin(["Con Heridos", "Con Muertos"])]
print(con_victimas.shape)

Gravedad_indicador_30d
Solo Daños     160018
Con Heridos    124529
Con Muertos      5489
Name: count, dtype: int64
(130018, 10)


Quitar los que no tienen coordenadas

In [43]:
print(con_victimas[["Latitud", "Longitud"]].isna().sum())
con_victimas = con_victimas.dropna(subset=["Latitud", "Longitud"])
print(con_victimas.shape)

# Revisar que no haya coordenadas por fuera de Bogota
fuera = con_victimas.query("Latitud < 3.7 or Latitud > 4.9 or Longitud < -74.5 or Longitud > -73.9")
print(len(fuera))

Latitud     4572
Longitud    4572
dtype: int64
(125446, 10)
0


Fecha, localidad y franja horaria

In [44]:
con_victimas["Fecha_Acc"] = pd.to_datetime(con_victimas["Fecha_Acc"])
con_victimas["Localidad"] = con_victimas["Localidad"].str.strip().str.upper()

def franja(hora):
    if hora <= 5:
        return "Madrugada (0-5)"
    elif hora <= 11:
        return "Mañana (6-11)"
    elif hora <= 17:
        return "Tarde (12-17)"
    else:
        return "Noche (18-23)"

con_victimas["franja"] = con_victimas["Hora_Acc"].apply(franja)
print(con_victimas["franja"].value_counts())

franja
Mañana (6-11)      40114
Tarde (12-17)      39490
Noche (18-23)      33072
Madrugada (0-5)    12770
Name: count, dtype: int64


In [45]:
repetidas = con_victimas.groupby(["Latitud", "Longitud"]).agg(siniestros=("Codigo_Accidente", "count"),
                                                               direcciones=("Direccion", "nunique")).reset_index()
print(repetidas.sort_values("siniestros", ascending=False).head(5))

malas = repetidas[repetidas["direcciones"] > 100][["Latitud", "Longitud"]]
print(malas)

con_victimas = con_victimas.merge(malas, on=["Latitud", "Longitud"], how="left", indicator=True)
con_victimas = con_victimas[con_victimas["_merge"] == "left_only"].drop(columns="_merge")
print(con_victimas.shape)

        Latitud   Longitud  siniestros  direcciones
43549  4.619318 -74.097908         292          287
54701  4.632361 -74.154053          49            2
59276  4.639739 -74.115770          44            4
90522  4.722221 -74.121347          42            4
47284  4.624301 -74.124142          40            4
        Latitud   Longitud
43549  4.619318 -74.097908
(125154, 11)


Revisar y guardar

In [46]:
print(con_victimas.groupby("AA_Acc").agg(cantidad=("Codigo_Accidente", "count")))

con_victimas.to_csv("../data/processed/siniestros.csv.gz", index=False)

        cantidad
AA_Acc          
2015       10226
2016       10330
2017       10189
2018       11895
2019       12151
2020        8820
2021       11467
2022       12595
2023       12999
2024       12909
2025       11573


## Decisiones
- Juntamos 2015-2024 y 2025 después de igualar los nombres de las columnas. Quedaron 290.187 siniestros.
- En 2025 las coordenadas venían como texto con coma. Las pasamos a número.
- 3 siniestros estaban repetidos entre los dos archivos. Dejamos una sola versión: 290.184.
- Nos quedamos con los siniestros con muertos o heridos (gravedad a 30 días): 130.018.
- Quitamos 4.572 sin coordenadas: quedan 125.446.
- Ninguna coordenada quedó por fuera de Bogotá.